In [ ]:
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve,
)
import matplotlib.pyplot as plt
from sklearn.datasets import 


ImportError: cannot import name 'pr_auc_curve' from 'sklearn.metrics' (/home/dinesh/miniconda3/envs/churn/lib/python3.13/site-packages/sklearn/metrics/__init__.py)

In [15]:
import numpy as np
np.random.randint(0, 10, 10)

array([0, 4, 0, 5, 7, 4, 5, 2, 1, 5])

In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.datasets import load_wine
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
)

# Name the experiment (shows up as a group in the MLflow UI)
mlflow.set_experiment("wine-classifier-tracking")

# Create a binary target for ROC/PR metrics
a = load_wine()
X = a.data
y = a.target

y_binary = (y > 1).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_binary, test_size=0.2, random_state=42
)

hyperparam_sets = [
    {"n_estimators": 50, "max_depth": 3},
    {"n_estimators": 100, "max_depth": 5},
    {"n_estimators": 200, "max_depth": None},
]

for params in hyperparam_sets:
    with mlflow.start_run(run_name=f"rf_n{params['n_estimators']}_d{params['max_depth']}"):
        mlflow.log_params(params)

        model = RandomForestClassifier(random_state=42, **params)
        model.fit(X_train, y_train)

        preds = model.predict(X_test)
        proba = model.predict_proba(X_test)[:, 1]

        accuracy = accuracy_score(y_test, preds)
        f1 = f1_score(y_test, preds)
        roc_auc = roc_auc_score(y_test, proba)
        pr_auc = average_precision_score(y_test, proba)

        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("f1_score", f1)
        mlflow.log_metric("roc_auc", roc_auc)
        mlflow.log_metric("pr_auc", pr_auc)

        mlflow.sklearn.log_model(model, "model")

        print(
            f"Run with {params} -> accuracy={accuracy:.3f}, "
            f"f1={f1:.3f}, roc_auc={roc_auc:.3f}, pr_auc={pr_auc:.3f}"
        )

print("\nDone. Run `mlflow ui` in this same directory, then open http://localhost:5000")


In [ ]:
"""
MLflow Experiment Tracking — Hands-On Example
Trains a RandomForest on the wine dataset with different hyperparameters,
logging each run to MLflow so you can compare them in the UI.
""